In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "mistral-small-latest",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='01526460-9d51-4074-b273-b76c60e82fd3'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'BuY16nhUX', 'type': 'function', 'function': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 97, 'total_tokens': 114, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedd3-2024-78d2-b7fd-5493b33484a5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'BuY16nhUX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 17, 'total_tokens': 114}),
              ToolMessage(

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='419e2ad7-6183-4520-bc39-85255d7e1fe5'),
              AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How are you?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 98, 'total_tokens': 128, 'completion_tokens': 30, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019eedd4-768d-7df2-b458-c8d8f9bbe704-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 30, 'total_tokens': 128})]}


## Read state

In [10]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "mistral-small-latest",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='a12029ec-f039-4457-afb2-78ea3a2f4d39'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'OpQfdTc52', 'type': 'function', 'function': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 145, 'total_tokens': 162, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedd5-3d41-75a3-bbf5-7076b4cdeb9f-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'OpQfdTc52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 17, 'total_tokens': 162}),
              ToolMessag

In [12]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='a12029ec-f039-4457-afb2-78ea3a2f4d39'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'OpQfdTc52', 'type': 'function', 'function': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 145, 'total_tokens': 162, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eedd5-3d41-75a3-bbf5-7076b4cdeb9f-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'OpQfdTc52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 17, 'total_tokens': 162}),
              ToolMessag